# Practical example: Monte Carlo simulations

In this notebook, we will set up simple MC simulations in the canonical ensemble.

## Canonical Ensemble

In [ ]:
from icet.core.cluster_expansion import ClusterExpansion
from pathlib import Path
import numpy as np
from tqdm import tqdm

# First, we load the desired model
chemical_symbols = ["Cu", "Ni"]

max_atom_num = 8

base_path = Path.cwd().parents[1]
struct_path = (
    base_path
    / "data"
    / f"CE_dataset_{chemical_symbols[0]}{chemical_symbols[1]}"
)

ce = ClusterExpansion.read(struct_path / f"ce_model_{max_atom_num}.ce")


## Run MC simulation in the Canonical ensemble

In [ ]:

from ase.build import make_supercell
from ase.visualize import view

# we start by making a supercell
structure = make_supercell(
    ce.primitive_structure, 3 * np.array([[-1, 1, 1], [1, -1, 1], [1, 1, -1]])
)

# then, we create some compositions and populate the supercell with random atoms
compositions = np.linspace(0, 1, 11)
structures = []
for composition in tqdm(compositions):
    cstructure = structure.copy()
    num_species_2 = int(composition * len(cstructure))
    random_indices = np.random.choice(
        len(cstructure), size=num_species_2, replace=False
    )
    cstructure.symbols[:] = chemical_symbols[0]
    # for random starting configuration
    # cstructure.symbols[random_indices] = chemical_symbols[1]
    # for non-random starting configuration
    cstructure.symbols[:num_species_2] = chemical_symbols[1]
    structures.append(cstructure)

view(structures[5], viewer="ngl")



In [ ]:
from mchammer.ensembles import CanonicalEnsemble
from mchammer.calculators import ClusterExpansionCalculator

# This contains more efficient implementations for CE in particular. E.g. if one atom changes, it will only compute changes for itself and the direct neighbors
# Important to note is that this sets up the calculator for a specific supercell size
calculator = ClusterExpansionCalculator(structure, ce)


We can also add observers to the Monte Carlo simulations to conveniently analyze specific properties.  Here, we will add an observer for the structure factor, which provides an indication for the long range order.

$S(\mathbf{q}) \propto \sum_{j,k}^{N}e^{-i\mathbf{q}\cdot(\mathbf{R}_k - \mathbf{R}_j)}$

There are also observers available for various other quantities. See an overview [here](https://icet.materialsmodeling.org/moduleref/observers.html).

In [ ]:
from mchammer.observers import StructureFactorObserver

qpoints = 2 * np.pi / structure.cell.lengths()[0] * np.eye(3)
SF_oberver = StructureFactorObserver(
    structures[5],
    q_points=qpoints,
    interval=len(structure),
)
print(qpoints)

In [ ]:
seed = np.random.randint(0, 1e5)
print("seed:", seed)
temperature = 300

traj_file = struct_path / f"MC_NVT_{compositions[1]}.dc"
if traj_file.exists():
    # delete trajectory file if it already exists
    traj_file.unlink()

mc = CanonicalEnsemble(
    structure=structures[5].copy(),
    calculator=calculator,
    temperature=temperature,
    dc_filename=traj_file,
    random_seed=seed,
    # trajectory_write_interval=1, # default is number of sites, if you set this to 1, you get the individual swaps
)

mc.attach_observer(SF_oberver)

mc.run(number_of_trial_steps=len(mc.structure) * 50)
view(mc.structure, viewer="ngl")

## Analyzing the trajectory

The DataContainer object contains the trajectory. It can be read from the trajectory file via `DataContainer.read(file_name)`.

TASKs:
- Try running MC for different temperatures, what do you discover?
- What is the difference when using with a less random or more ordered starting configuration?
- How large can you make the structure until you run into performance issues?

In [ ]:
from mchammer import DataContainer
import matplotlib.pyplot as plt

# you can get the data container from the mc object
# dc = mc.data_container
# when reading the file, it is essential to convert it to a string first
dc = DataContainer.read(str(traj_file.resolve()))
energy = dc.get("potential", start=0) / len(structures[1])
acceptance_ratio = dc.get("acceptance_ratio", start=0)
f, axs = plt.subplots(2,1)
concentrations = []

axs[0].plot(acceptance_ratio)
axs[0].set_xlabel("step")
axs[0].set_ylabel("acceptance ratio")

axs[1].plot(energy)
axs[1].set_xlabel("step")
axs[1].set_ylabel("Energy / eV/atom")

# this might break for large supercells
traj = dc.get_trajectory()
view(traj)

## Investigating other observables

You can check `dc.observables` for the names of any tracked observable. In the following we will plot the structure factors we computed above.

In [ ]:

for key in dc.observables:
    print(key)
    if "sfo" in key:
        plt.plot(dc.get(key, start=0), label=key)
        plt.xlabel("step")
        plt.ylabel("structure factor")
        plt.legend()

However, we usually cannot gather much information from the values of the observables at specific steps. Interesting properties typically emerge from the average after an equilibration period for different starting settings (concentrations/temperatures etc.)


TASK:
- perform MC simulations for a range of compositions (code below) and plot the resulting energies and acceptance ratios - what do you see?
- what difference do you see at different temperatures?

In [ ]:
# we set up some dictionaries (only call this once to not overwrite temperatures)
aratios_T = {}
energies_T = {}
sfos_T = {}

In [ ]:
aratios = []
energies = []
sfos = []
equil_steps = 10
temperature = 300
for sid in range(1, len(compositions) - 1):
    seed = np.random.randint(0, 1e5)
    print("seed:", seed)

    traj_file = struct_path / f"MC_NVT_{compositions[sid]}.dc"
    if traj_file.exists():
        # delete trajectory file if it already exists
        traj_file.unlink()

    mc = CanonicalEnsemble(
        structure=structures[sid].copy(),
        calculator=calculator,
        temperature=temperature,
        dc_filename=traj_file,
        random_seed=seed,
    )

    mc.attach_observer(SF_oberver)

    mc.run(number_of_trial_steps=len(mc.structure) * 50)
    dc = mc.data_container
    energy = dc.get("potential", start=equil_steps) / len(structures[sid])
    acceptance_ratio = dc.get("acceptance_ratio", start=equil_steps)
    aratios.append(acceptance_ratio)
    energies.append(energy)
    sfos_q = []
    for key in dc.observables:
        if f"sfo_{chemical_symbols[0]}_{chemical_symbols[0]}" in key:
            sfos_q.append(dc.get(key, start=equil_steps))
    sfos.append(np.mean(sfos_q, axis=0))

aratios_T[temperature] = np.asarray(aratios)
energies_T[temperature] = np.asarray(energies)
sfos_T[temperature] = np.asarray(sfos)

In [ ]:
for temperature in aratios_T.keys():
    plt.figure(1)
    plt.plot(
        compositions[1:-1],
        np.mean(energies_T[temperature], axis=1),
        label=f"{temperature} K",
    )
    plt.xlabel(f"{chemical_symbols[1]} fraction")
    plt.ylabel("Energy / eV/atom")
    plt.legend()
    plt.figure(2)
    plt.plot(
        compositions[1:-1],
        np.mean(aratios_T[temperature], axis=1),
        label=f"{temperature} K",
    )
    plt.xlabel(f"{chemical_symbols[1]} fraction")
    plt.ylabel("acceptance ratio")
    plt.legend()
    plt.figure(3)
    plt.plot(
        compositions[1:-1],
        np.mean(sfos_T[temperature], axis=1),
        label=f"{temperature} K",
    )
    plt.xlabel(f"{chemical_symbols[1]} fraction")
    plt.ylabel(f"{chemical_symbols[0]} structure factor")
    plt.legend()